# Feature engineering

This notebook fits all learned feature transformations on gold training data only. It writes one unified artifact per split containing every feature representation required by downstream models and explanations.


In [1]:
import importlib.util
import subprocess
import sys

required_packages = {
    "pandas": "pandas>=2.1.0",
    "pyspark": "pyspark>=3.5.0,<4.0",
}
missing_packages = [
    package
    for module, package in required_packages.items()
    if importlib.util.find_spec(module) is None
]
if missing_packages:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", *missing_packages],
        check=True,
    )
    importlib.invalidate_caches()

import json
import os
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
from pyspark.ml.feature import (
    CountVectorizer, HashingTF, IDF, MinHashLSH, MinMaxScaler,
    StandardScaler, VectorAssembler, Word2Vec
)
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import ArrayType, StringType
from pyspark import StorageLevel

SEED = 42
KAGGLE_SAFE_MODE = os.getenv("KAGGLE_SAFE_MODE", "1") == "1"
ENABLE_BANGLISH_NEAR_DUPLICATES = os.getenv("ENABLE_BANGLISH_NEAR_DUPLICATES", "0") == "1"
BASE_DIR = Path("/kaggle/working/bangla_fake_review")
PROCESSED_DIR = BASE_DIR / "processed"
MODELS_DIR = BASE_DIR / "models"
AUDIT_DIR = BASE_DIR / "audit"
EDA_DIR = BASE_DIR / "eda"
for directory in (MODELS_DIR, AUDIT_DIR, EDA_DIR):
    directory.mkdir(parents=True, exist_ok=True)

spark = (
    SparkSession.builder
    .appName("BanglaFakeReview_Features")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.shuffle.partitions", "24")
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

def resolve_input_directory(
    preferred, required_filenames, input_root=Path("/kaggle/input")
):
    if all((preferred / name).exists() for name in required_filenames):
        return preferred
    candidates = []
    if input_root.exists():
        for first_file in input_root.rglob(required_filenames[0]):
            candidate = first_file.parent
            if all((candidate / name).exists() for name in required_filenames):
                candidates.append(candidate)
    candidates = sorted(set(candidates))
    if not candidates:
        raise FileNotFoundError(
            "Attach the saved output of notebook 02 as a Kaggle input before "
            "running notebook 03."
        )
    if len(candidates) > 1:
        raise RuntimeError(f"Multiple notebook 02 outputs found: {candidates}")
    return candidates[0]

stage_two_files = [
    "train_features.parquet",
    "validation_features.parquet",
    "test_features.parquet",
    "banglish_features.parquet",
]
INPUT_PROCESSED_DIR = resolve_input_directory(PROCESSED_DIR, stage_two_files)
print(f"Reading notebook 02 artifacts from {INPUT_PROCESSED_DIR}")

datasets = {
    "train": spark.read.parquet(str(INPUT_PROCESSED_DIR / "train_features.parquet")),
    "validation": spark.read.parquet(str(INPUT_PROCESSED_DIR / "validation_features.parquet")),
    "test": spark.read.parquet(str(INPUT_PROCESSED_DIR / "test_features.parquet")),
    "banglish": spark.read.parquet(str(INPUT_PROCESSED_DIR / "banglish_features.parquet")),
}
train = datasets["train"]


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/20 19:12:43 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Reading notebook 02 artifacts from /kaggle/input/notebooks/borshon47/preprocess1/bangla_fake_review/processed


In [2]:
# Fit unsupervised representations on the full transductive corpus.
# Keep only vectorizer inputs so BFRD-only audit columns do not block unioning
# with the unlabeled Banglish schema.
fit_columns = ["review_id", "model_text", "tokens"]
fit_corpus = (
    datasets["train"].select(*fit_columns)
    .unionByName(datasets["validation"].select(*fit_columns))
    .unionByName(datasets["test"].select(*fit_columns))
    .unionByName(datasets["banglish"].select(*fit_columns))
    .dropDuplicates(["review_id"])
    .persist(StorageLevel.DISK_ONLY)
)
GLOBAL_MIN_DF = 5.0
GLOBAL_FIT_ROWS = fit_corpus.count()
count_vectorizer = CountVectorizer(
    inputCol="tokens",
    outputCol="term_frequency",
    vocabSize=30000 if KAGGLE_SAFE_MODE else 100000,
    minDF=GLOBAL_MIN_DF,
)
count_vectorizer_model = count_vectorizer.fit(fit_corpus)
train_term_frequency = count_vectorizer_model.transform(train)

idf = IDF(
    inputCol="term_frequency",
    outputCol="tfidf",
    minDocFreq=int(GLOBAL_MIN_DF),
)
idf_model = idf.fit(count_vectorizer_model.transform(fit_corpus))

word2vec = Word2Vec(
    inputCol="tokens",
    outputCol="word2vec",
    vectorSize=25 if KAGGLE_SAFE_MODE else 100,
    minCount=3,
    maxIter=5 if KAGGLE_SAFE_MODE else 10,
    numPartitions=4,
    seed=SEED,
)
word2vec_model = word2vec.fit(fit_corpus)

count_vectorizer_model.write().overwrite().save(
    str(MODELS_DIR / "count_vectorizer")
)
idf_model.write().overwrite().save(str(MODELS_DIR / "idf"))
word2vec_model.write().overwrite().save(str(MODELS_DIR / "word2vec"))
fit_corpus.unpersist(blocking=True)


26/09/20 19:17:24 WARN TaskSetManager: Stage 49 contains a task of very large size (1912 KiB). The maximum recommended task size is 1000 KiB.


DataFrame[review_id: string, model_text: string, tokens: array<string>]

In [3]:
@F.udf(ArrayType(StringType()))
def token_shingles(tokens):
    values = [str(token) for token in (tokens or []) if str(token)]
    if len(values) < 3:
        return ["__short__:" + "_".join(values)] if values else ["__empty__"]
    return [
        "_".join(values[index:index + 3])
        for index in range(len(values) - 2)
    ]


hashing = HashingTF(
    inputCol="token_shingles",
    outputCol="shingle_vector",
    numFeatures=8192,
    binary=True,
)

def add_shingle_vector(frame):
    return hashing.transform(
        frame.withColumn("token_shingles", token_shingles("tokens"))
    )

train_shingles = add_shingle_vector(train)
minhash_model = MinHashLSH(
    inputCol="shingle_vector",
    outputCol="minhash_signature",
    numHashTables=3,
    seed=SEED,
).fit(train_shingles)
minhash_model.write().overwrite().save(str(MODELS_DIR / "minhash"))

reference = minhash_model.transform(train_shingles).select(
    "review_id", "shingle_vector", "minhash_signature"
)

def add_near_duplicate_features(frame, self_comparison=False):
    query = minhash_model.transform(add_shingle_vector(frame)).select(
        "review_id", "shingle_vector", "minhash_signature"
    )
    pairs = minhash_model.approxSimilarityJoin(
        query,
        reference,
        threshold=0.30,
        distCol="jaccard_distance",
    )
    if self_comparison:
        pairs = pairs.filter(
            F.col("datasetA.review_id") != F.col("datasetB.review_id")
        )
    pair_summary = (
        pairs.select(
            F.col("datasetA.review_id").alias("review_id"),
            F.col("datasetB.review_id").alias("neighbor_id"),
            (1.0 - F.col("jaccard_distance")).alias("similarity"),
        )
        .dropDuplicates(["review_id", "neighbor_id"])
        .groupBy("review_id")
        .agg(
            F.count("*").cast("double").alias("near_duplicate_degree"),
            F.max("similarity").alias("near_duplicate_similarity"),
        )
    )
    return (
        frame.join(pair_summary, "review_id", "left")
        .fillna({
            "near_duplicate_degree": 0.0,
            "near_duplicate_similarity": 0.0,
        })
        .withColumn(
            "near_duplicate_cluster_size",
            F.col("near_duplicate_degree") + F.lit(1.0),
        )
    )

def add_zero_near_duplicate_features(frame):
    return (
        frame.withColumn("near_duplicate_degree", F.lit(0.0))
        .withColumn("near_duplicate_similarity", F.lit(0.0))
        .withColumn("near_duplicate_cluster_size", F.lit(1.0))
    )

near_datasets = {}
for name, frame in datasets.items():
    if name == "banglish" and not ENABLE_BANGLISH_NEAR_DUPLICATES:
        near_datasets[name] = add_zero_near_duplicate_features(frame)
    else:
        near_datasets[name] = add_near_duplicate_features(
            frame,
            self_comparison=(name == "train"),
        )


In [4]:
HANDCRAFTED_FEATURES = [
    "codemix_ratio", "repeated_char_ratio",
    "url_count", "email_count", "phone_count", "price_count",
    "hashtag_count", "mention_count", "exclamation_count", "emoji_count",
    "promo_keyword_hits", "bengali_chars", "latin_chars", "digit_chars",
    "other_chars", "exact_dup_cluster_size", "is_exact_duplicate",
    "near_duplicate_degree", "near_duplicate_similarity",
    "near_duplicate_cluster_size", "has_url", "has_email", "has_phone",
    "has_price", "has_promo", "has_emoji",
]
NO_NEAR_DUPLICATE_FEATURES = [
    feature for feature in HANDCRAFTED_FEATURES
    if not feature.startswith("near_duplicate_")
]

handcrafted_assembler = VectorAssembler(
    inputCols=HANDCRAFTED_FEATURES,
    outputCol="handcrafted_raw",
)
no_near_assembler = VectorAssembler(
    inputCols=NO_NEAR_DUPLICATE_FEATURES,
    outputCol="handcrafted_no_near_raw",
)

train_assembled = no_near_assembler.transform(
    handcrafted_assembler.transform(near_datasets["train"])
)
handcrafted_scaler_model = StandardScaler(
    inputCol="handcrafted_raw",
    outputCol="handcrafted_scaled",
    withMean=False,
    withStd=True,
).fit(train_assembled)
no_near_scaler_model = StandardScaler(
    inputCol="handcrafted_no_near_raw",
    outputCol="handcrafted_no_near_scaled",
    withMean=False,
    withStd=True,
).fit(train_assembled)
naive_bayes_scaler_model = MinMaxScaler(
    inputCol="handcrafted_raw",
    outputCol="handcrafted_nonnegative",
).fit(train_assembled)

handcrafted_scaler_model.write().overwrite().save(
    str(MODELS_DIR / "handcrafted_scaler")
)
no_near_scaler_model.write().overwrite().save(
    str(MODELS_DIR / "no_near_scaler")
)
naive_bayes_scaler_model.write().overwrite().save(
    str(MODELS_DIR / "naive_bayes_scaler")
)

linear_assembler = VectorAssembler(
    inputCols=["tfidf", "handcrafted_scaled"],
    outputCol="features_linear",
)
linear_no_near_assembler = VectorAssembler(
    inputCols=["tfidf", "handcrafted_no_near_scaled"],
    outputCol="features_linear_no_near",
)
tree_assembler = VectorAssembler(
    inputCols=["word2vec", "handcrafted_raw"],
    outputCol="features_tree",
)
naive_bayes_assembler = VectorAssembler(
    inputCols=["term_frequency", "handcrafted_nonnegative"],
    outputCol="features_nb",
)

def build_feature_matrix(frame):
    transformed = count_vectorizer_model.transform(frame)
    transformed = idf_model.transform(transformed)
    transformed = word2vec_model.transform(transformed)
    transformed = handcrafted_assembler.transform(transformed)
    transformed = no_near_assembler.transform(transformed)
    transformed = handcrafted_scaler_model.transform(transformed)
    transformed = no_near_scaler_model.transform(transformed)
    transformed = naive_bayes_scaler_model.transform(transformed)
    transformed = linear_assembler.transform(transformed)
    transformed = linear_no_near_assembler.transform(transformed)
    transformed = tree_assembler.transform(transformed)
    transformed = naive_bayes_assembler.transform(transformed)
    return transformed.withColumn("features_tfidf_only", F.col("tfidf"))

required_columns = {
    "review_id", "safe_review", "model_text", "tokens", "source_dataset",
    "source_split", "label_source", "features_linear", "features_tree",
    "features_nb", "features_tfidf_only", "features_linear_no_near",
    "handcrafted_raw", "handcrafted_scaled", "handcrafted_nonnegative",
}

# Persist only columns consumed by notebooks 04-06. Dropping shingle and
# intermediate columns substantially reduces Parquet memory pressure.
output_columns = [
    "review_id", "safe_review", "model_text", "tokens",
    "source_dataset", "source_split", "label_source",
    "label", "confidence_score",
    "char_length", "token_count", "exact_dup_cluster_size",
    "is_exact_duplicate", "codemix_ratio", "repeated_char_ratio",
    "url_count", "email_count", "phone_count", "price_count",
    "hashtag_count", "mention_count", "exclamation_count",
    "emoji_count", "promo_keyword_hits", "bengali_chars",
    "latin_chars", "digit_chars", "other_chars",
    "has_url", "has_email", "has_phone", "has_price",
    "has_promo", "has_emoji",
    "term_frequency", "tfidf", "word2vec",
    "near_duplicate_degree", "near_duplicate_similarity",
    "near_duplicate_cluster_size",
    "features_linear", "features_tree", "features_nb",
    "features_tfidf_only", "features_linear_no_near",
    "handcrafted_raw", "handcrafted_scaled",
    "handcrafted_nonnegative",
]
for name in ["train", "validation", "test", "banglish"]:
    frame = build_feature_matrix(near_datasets[name]).select(*output_columns)
    frame = frame.persist(StorageLevel.DISK_ONLY)
    missing = sorted(required_columns.difference(frame.columns))
    if missing:
        raise AssertionError(f"{name} is missing columns: {missing}")
    # coalesce avoids a full shuffle before the write.
    frame.coalesce(8).write.mode("overwrite").parquet(
        str(PROCESSED_DIR / f"{name}_model_ready.parquet")
    )
    frame.unpersist(blocking=True)
    spark.catalog.clearCache()


26/09/20 19:17:27 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/09/20 19:18:06 WARN DAGScheduler: Broadcasting large task binary with size 5.7 MiB
26/09/20 19:18:11 WARN DAGScheduler: Broadcasting large task binary with size 9.2 MiB
26/09/20 19:18:24 WARN DAGScheduler: Broadcasting large task binary with size 5.7 MiB
26/09/20 19:18:27 WARN DAGScheduler: Broadcasting large task binary with size 9.2 MiB
26/09/20 19:18:36 WARN DAGScheduler: Broadcasting large task binary with size 5.7 MiB
26/09/20 19:18:40 WARN DAGScheduler: Broadcasting large task binary with size 9.2 MiB
26/09/20 19:18:44 WARN DAGScheduler: Broadcasting large task binary with size 9.2 MiB


In [5]:
near_duplicate_summary = []
for name in ["train", "validation", "test", "banglish"]:
    frame = spark.read.parquet(str(PROCESSED_DIR / f"{name}_model_ready.parquet"))
    row = frame.agg(
        F.count("*").alias("rows"),
        F.avg("near_duplicate_degree").alias("mean_degree"),
        F.max("near_duplicate_degree").alias("maximum_degree"),
        F.avg("near_duplicate_similarity").alias("mean_similarity"),
    ).first()
    near_duplicate_summary.append({
        "dataset": name,
        "rows": int(row["rows"]),
        "mean_degree": float(row["mean_degree"] or 0.0),
        "maximum_degree": float(row["maximum_degree"] or 0.0),
        "mean_similarity": float(row["mean_similarity"] or 0.0),
    })
    frame.unpersist(blocking=True)

pd.DataFrame(near_duplicate_summary).to_csv(
    EDA_DIR / "near_duplicate_summary.csv",
    index=False,
)

audit_train = spark.read.parquet(str(PROCESSED_DIR / "train_model_ready.parquet"))
sample_linear = audit_train.select("features_linear").first()[0]
sample_tree = audit_train.select("features_tree").first()[0]
sample_nb = audit_train.select("features_nb").first()[0]
audit_train.unpersist(blocking=True)

feature_audit = {
    "created_at": datetime.now(timezone.utc).isoformat(),
    "fit_scope": "global unlabeled corpus: all BFRD splits plus BanglishRev; supervised models remain train-only",
    "global_fit_rows": GLOBAL_FIT_ROWS,
    "global_min_doc_frequency": GLOBAL_MIN_DF,
    "kaggle_safe_mode": KAGGLE_SAFE_MODE,
    "count_vectorizer_vocabulary": len(count_vectorizer_model.vocabulary),
    "word2vec_dimensions": 25 if KAGGLE_SAFE_MODE else 100,
    "minhash": {
        "token_shingle_size": 3,
        "distance_threshold": 0.30,
        "hash_tables": 3,
        "hash_dimensions": 8192,
        "banglish_near_duplicates_enabled": ENABLE_BANGLISH_NEAR_DUPLICATES,
    },
    "feature_dimensions": {
        "linear": sample_linear.size,
        "tree": sample_tree.size,
        "naive_bayes": sample_nb.size,
    },
    "handcrafted_features": HANDCRAFTED_FEATURES,
    "near_duplicate_summary": near_duplicate_summary,
}
with open(AUDIT_DIR / "feature_audit.json", "w", encoding="utf-8") as handle:
    json.dump(feature_audit, handle, ensure_ascii=False, indent=2)

print(json.dumps(feature_audit, ensure_ascii=False, indent=2))


26/09/20 19:19:16 WARN DAGScheduler: Broadcasting large task binary with size 1621.1 KiB
26/09/20 19:19:17 WARN DAGScheduler: Broadcasting large task binary with size 1884.9 KiB


{
  "created_at": "2026-09-20T19:19:17.260419+00:00",
  "fit_scope": "global unlabeled corpus: all BFRD splits plus BanglishRev; supervised models remain train-only",
  "global_fit_rows": 331987,
  "global_min_doc_frequency": 5.0,
  "kaggle_safe_mode": true,
  "count_vectorizer_vocabulary": 30000,
  "word2vec_dimensions": 25,
  "minhash": {
    "token_shingle_size": 3,
    "distance_threshold": 0.3,
    "hash_tables": 3,
    "hash_dimensions": 8192,
    "banglish_near_duplicates_enabled": false
  },
  "feature_dimensions": {
    "linear": 30026,
    "tree": 51,
    "naive_bayes": 30026
  },
  "handcrafted_features": [
    "codemix_ratio",
    "repeated_char_ratio",
    "url_count",
    "email_count",
    "phone_count",
    "price_count",
    "hashtag_count",
    "mention_count",
    "exclamation_count",
    "emoji_count",
    "promo_keyword_hits",
    "bengali_chars",
    "latin_chars",
    "digit_chars",
    "other_chars",
    "exact_dup_cluster_size",
    "is_exact_duplicate",
    "n